In [ ]:
from typing import Dict
from data import load_data, time_train_test_split
from config import path_str, TARGET_COL, FEATURE_LEVEL_CONFIGS
from evaluation import eval_best_config_on_holdout
from training import SortFeaturesByCorrElastic, SortFeaturesByImportanceXGB


import pandas as pd
import numpy as np


import warnings
warnings.filterwarnings("error", category=FutureWarning)

df_raw = load_data(path_str.TRAIN_DIR)
df_cut_raw = df_raw[1006:]

X = df_cut_raw.copy()
y = df_cut_raw[TARGET_COL]


X_train, X_test, y_train, y_test = time_train_test_split(X, y)


FEATURE_RANKINGS: Dict[str, list] = {}

for mt in ["ols", "ridge", "lasso", "elastic", "lgbm", "xgb"]:
    if mt in ["ols", "ridge", "lasso", "elastic"]:
        FEATURE_RANKINGS[mt] = SortFeaturesByCorrElastic(X_train, y_train)
    elif mt in ["xgb", "lgbm"]:
        FEATURE_RANKINGS[mt] = SortFeaturesByImportanceXGB(X_train, y_train)
    else:
        FEATURE_RANKINGS[mt] = []


MODEL_TYPES = ["ols", "ridge", "lasso", "elastic", "lgbm", "xgb"]
LABELS = ["final", "prune", "broad_tests"]
RESULTS_DIR = "optuna_results"

all_results = []

for label in LABELS:
    for config in FEATURE_LEVEL_CONFIGS:
        for model_type in MODEL_TYPES:
            print(f"Evaluating {model_type} ({label}) on holdout...")
            try:
                res = eval_best_config_on_holdout(
                    model_type=model_type,
                    label=label,
                    X=X,
                    y=y,
                    feature_rankings=FEATURE_RANKINGS[model_type],
                    results_dir=RESULTS_DIR,
                    feat_level=config,
                    test_frac=0.2,
                )
                all_results.append(res)
            except FileNotFoundError:
                print(f"No trials file for {model_type} ({label}), skipping.")
            except Exception as e:
                print(f"[ERROR] {model_type} ({label}): {e}")


results_df = pd.DataFrame(all_results)
results_df.sort_values(["r2_holdout"], ascending=False, inplace=True)
print(results_df)

pivot_r2 = results_df.pivot_table(
    index="model_type",
    columns=["label", "feature_level"],
    values="r2_holdout",
    aggfunc="max",   # or "mean" if you ever duplicate exact combos
)

pivot_r2.to_csv("pivot_r2.csv")
print(pivot_r2)



c:\Users\lhkke\Documents\HullTactical\HullTactical\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Evaluating ols (final) on holdout...
Loading trials from: optuna_results\ols_none_final_trials.csv
Evaluating ridge (final) on holdout...
Loading trials from: optuna_results\ridge_none_final_trials.csv
Evaluating lasso (final) on holdout...
Loading trials from: optuna_results\lasso_none_final_trials.csv
Evaluating elastic (final) on holdout...
Loading trials from: optuna_results\elastic_none_final_trials.csv
Evaluating lgbm (final) on holdout...
Loading trials from: optuna_results\lgbm_none_final_trials.csv
Evaluating xgb (final) on holdout...
Loading trials from: optuna_results\xgb_none_final_trials.csv
Evaluating ols (final) on holdout...
Loading trials from: optuna_results\ols_simple_final_trials.csv
Evaluating ridge (final) on holdout...
Loading trials from: optuna_results\ridge_simple_final_trials.csv
Evaluating lasso (final) on holdout...
Loading trials from: optuna_results\lasso_simple_final_trials.csv
Evaluating elastic (final) on holdout...
Loading trials from: optuna_results\

ValueError: Index contains duplicate entries, cannot reshape